In [1]:
from env import config, BASE_DIR
from sqlalchemy import create_engine, text

In [2]:
import pandas as pd
import datetime

In [3]:
DATABASE_URL = config("FLIGHT_PRICES_DATABASE_URL", cast=str, default=None)
DATABASE_URL is not None

True

In [4]:
engine = create_engine(str(DATABASE_URL))

In [5]:
DATASET_DIR = BASE_DIR / "datasets"
DATASET_DIR.exists()

True

In [6]:
GRETEL_DIR = DATASET_DIR / "gretel"
GRETEL_DIR.exists()

True

In [7]:
table_name = "flight_prices"
df = pd.read_sql(table_name, engine, index_col = "index")
df.tail(n = 1)
# df = out_df.to_sql(table_name, engine, if_exists='replace', index=True)

,flightDate,startingAirport,destinationAirport,isBasicEconomy,isRefundable,isNonStop,segmentsAirlineName,totalFare
index,,,,,,,,
9591,2022-04-22,IAD,LAX,1,0,1,United,221


In [8]:
last_row = df.tail(n=1)
last_idx = last_row.index[0]
last_idx

np.int64(9591)

In [9]:
max_flight_date = df.flightDate.max()
max_flight_date

Timestamp('2022-04-22 00:00:00')

In [21]:
import csv
import glob
import os 

gretel_dfs = []
csvfiles = glob.glob(os.path.join(GRETEL_DIR, '*.csv'))

for csvfile in csvfiles:
    df = pd.read_csv(csvfile)
    gretel_dfs.append(df)

new_df = pd.concat(gretel_dfs)
new_df.index = new_df.index + last_idx + 1
new_df['flightDate'] = pd.to_datetime(new_df['flightDate'])
new_df['flightDate'] = new_df['flightDate'].apply(lambda x: max_flight_date + (max_flight_date - x) + datetime.timedelta(days=1))
new_df.head()

,flightDate,startingAirport,destinationAirport,isBasicEconomy,isRefundable,isNonStop,segmentsAirlineName,totalFare
9592,2022-04-23,SFO,IAD,0,0,1,United,342
9593,2022-04-25,EWR,DEN,0,0,1,American Airlines,217
9594,2022-04-28,EWR,ORD,0,0,0,American Airlines||American Airlines,286
9595,2022-04-27,SFO,CLT,0,0,1,American Airlines,255
9596,2022-04-27,EWR,OAK,0,0,0,United||United||Delta,676


In [22]:
new_df.flightDate.max().date()

datetime.date(2022, 4, 28)

In [23]:
new_df.to_sql(table_name, engine, if_exists='append', index=True)

10000

In [25]:
df = pd.read_sql(table_name, engine, index_col="index")
df.tail()

,flightDate,startingAirport,destinationAirport,isBasicEconomy,isRefundable,isNonStop,segmentsAirlineName,totalFare
index,,,,,,,,
14587,2022-04-24,CLT,IAD,1,0,0,Delta||Delta,420
14588,2022-04-28,SFO,BOS,0,0,0,Delta||Delta||Delta,360
14589,2022-04-24,DFW,EWR,0,0,0,Spirit Airlines||Spirit Airlines,331
14590,2022-04-23,ATL,ATL,0,0,0,American Airlines||American Airlines,455
14591,2022-04-27,ORD,JFK,0,0,0,Delta||Delta||Delta,321
